In [0]:
dbutils.widgets.dropdown(
    name="environment",
    defaultValue="dev",
    choices=["dev","prd","qa"],
    label="select Environment"
)
env = dbutils.widgets.get("environment")

goldTablName   = f"saleslake_{env}.gold_{env}.refinedinvoice"
silverTablName = f"saleslake_{env}.silver_{env}.cleanedinvoice"
print(f"silver: {silverTablName}\ngold  : {goldTablName}")

In [0]:
spark.sql(f"""

MERGE INTO {goldTablName} tgt
USING (

    WITH latest_inv_silver AS (
        SELECT *
        FROM {silverTablName}
    ),

    latest_rm_dup_silver AS (
        SELECT *
        FROM (
            SELECT *,
                   ROW_NUMBER() OVER (
                       PARTITION BY invoice_id
                       ORDER BY ingest_ts DESC
                   ) AS rn
            FROM latest_inv_silver
        ) tmp
        WHERE rn = 1
    ),

    silver_gold_rec AS (
        SELECT
            s.*,
            g.invoice_id           AS g_invoice_id,

            CASE
                WHEN g.invoice_id IS NULL THEN 'NEW'

                WHEN NOT (s.subtotal_amount  <=> g.subtotal_amount)
                  OR NOT (s.discount_code    <=> g.discount_code)
                  OR NOT (s.discount_amount  <=> g.discount_amount)
                  OR NOT (s.tax_amount       <=> g.tax_amount)
                  OR NOT (s.total_amount     <=> g.total_amount)
                  OR NOT (s.payment_status   <=> g.payment_status)
                  OR NOT (s.payment_method   <=> g.payment_method)
                  OR NOT (s.payment_date     <=> g.payment_date)
                  OR NOT (s.due_date         <=> g.due_date)
                  OR NOT (s.customer_id      <=> g.customer_id)
                  OR NOT (s.region           <=> g.region)
                  OR NOT (s.store_id         <=> g.store_id)
                  OR NOT (s.channel          <=> g.channel)

                THEN 'CHANGE'

                ELSE 'NO_CHANGE'
            END AS rec_flag

        FROM latest_rm_dup_silver s
        LEFT JOIN {goldTablName} g
        ON s.invoice_id = g.invoice_id
    ),

    final_src AS (

        -- 🔴 Records to expire (old version)
        SELECT
            invoice_id AS merge_key,
            'UPDATE'   AS merge_flag,
            *
        FROM silver_gold_rec
        WHERE rec_flag = 'CHANGE'

        UNION ALL

        -- 🟢 Records to insert (new + changed)
        SELECT
            NULL       AS merge_key,
            'INSERT'   AS merge_flag,
            *
        FROM silver_gold_rec
        WHERE rec_flag IN ('NEW', 'CHANGE')

    )

    SELECT * FROM final_src

) src

ON tgt.invoice_id = src.merge_key

-- ✅ Update existing record
WHEN MATCHED AND src.merge_flag = 'UPDATE' THEN
UPDATE SET
    tgt.invoice_number = src.invoice_number,
    tgt.customer_id = src.customer_id,
    tgt.invoice_date = src.invoice_date,
    tgt.due_date = src.due_date,
    tgt.subtotal_amount = src.subtotal_amount,
    tgt.discount_code = src.discount_code,
    tgt.discount_amount = src.discount_amount,
    tgt.tax_amount = src.tax_amount,
    tgt.total_amount = src.total_amount,
    tgt.payment_status = src.payment_status,
    tgt.payment_method = src.payment_method,
    tgt.payment_date = src.payment_date,
    tgt.currency = src.currency,
    tgt.region = src.region,
    tgt.store_id = src.store_id,
    tgt.channel = src.channel,
    tgt.created_by = src.created_by,
    tgt.last_updt_ts = current_timestamp()

-- ✅ Insert new record
WHEN NOT MATCHED AND src.merge_flag = 'INSERT' THEN
INSERT (
    invoice_id,
    invoice_number,
    customer_id,
    invoice_date,
    due_date,
    subtotal_amount,
    discount_code,
    discount_amount,
    tax_amount,
    total_amount,
    payment_status,
    payment_method,
    payment_date,
    currency,
    region,
    store_id,
    channel,
    created_by,
    initial_load_ts,        
    last_updt_ts  

)
VALUES (
    src.invoice_id,
    src.invoice_number,
    src.customer_id,
    src.invoice_date,
    src.due_date,
    src.subtotal_amount,
    src.discount_code,
    src.discount_amount,
    src.tax_amount,
    src.total_amount,
    src.payment_status,
    src.payment_method,
    src.payment_date,
    src.currency,
    src.region,
    src.store_id,
    src.channel,
    src.created_by,
    current_timestamp(),    
    current_timestamp()    

)

""")

In [0]:
%sql
SELECT * FROM saleslake_dev.gold_dev.refinedinvoice;
SELECT * FROM saleslake_dev.silver_dev.cleanedinvoice;

In [0]:
%sql
describe extended saleslake_dev.gold_dev.refinedinvoice;
describe extended saleslake_dev.silver_dev.cleanedinvoice;

In [0]:
%sql
describe extended saleslake_dev.gold_dev.refinedinvoice;

In [0]:
%sql
SELECT * FROM  saleslake_dev.gold_dev.refinedinvoice;